# Section 1: Split and baseline

In [1]:
import sys
sys.path.append('../src')

from data import load_hour_data, chronological_split, get_xy

In [2]:
df = load_hour_data()

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (17379, 17)

First 5 rows:


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [3]:
train, test = chronological_split(df)

print("Training rows:", len(train))
print("Testing rows:", len(test))

print("\nTraining period:")
print("From:", train['dteday'].min())
print("To:", train['dteday'].max())

print("\nTesting period:")
print("From:", test['dteday'].min())
print("To:", test['dteday'].max())

Training rows: 13003
Testing rows: 4376

Training period:
From: 2011-01-01 00:00:00
To: 2012-06-30 00:00:00

Testing period:
From: 2012-07-01 00:00:00
To: 2012-12-31 00:00:00


### Why use a chronological split?

This dataset contains bike rental observations over time, so the model should learn from past data and predict future data. A random split could place future observations in the training set while earlier observations are in the test set, which would not represent a real forecasting situation. Therefore, the data is split chronologically at 2012-07-01, with earlier observations used for training and later observations used for testing.

In [4]:
X_train, y_train = get_xy(train)
X_test, y_test = get_xy(test)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nX_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (13003, 12)
y_train shape: (13003,)

X_test shape: (4376, 12)
y_test shape: (4376,)


### Baseline Linear Regression

For the baseline model, I use four simple features: hour of day (`hr`), temperature (`temp`), whether it is a working day (`workingday`), and year (`yr`). This gives us a simple starting model that later models can be compared against.

In [5]:
baseline_features = ['hr', 'temp', 'workingday', 'yr']

X_train_baseline = X_train[baseline_features]
X_test_baseline = X_test[baseline_features]

display(X_train_baseline.head())

,hr,temp,workingday,yr
0,0,0.24,0,0
1,1,0.22,0,0
2,2,0.22,0,0
3,3,0.24,0,0
4,4,0.24,0,0


In [6]:
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

baseline_model.fit(X_train_baseline, y_train)

baseline_predictions = baseline_model.predict(X_test_baseline)

print("Baseline model trained successfully.")
print("Number of predictions:", len(baseline_predictions))

Baseline model trained successfully.
Number of predictions: 4376


In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
mae = mean_absolute_error(y_test, baseline_predictions)
r2 = r2_score(y_test, baseline_predictions)

print("Baseline Linear Regression Results")
print("----------------------------------")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

Baseline Linear Regression Results
----------------------------------
RMSE: 191.29796121072496
MAE : 145.32068901300278
R2  : 0.24178549551517614


### Random Split Demo — NOT the headline result

The chronological split above is the main evaluation because this is time-based data. The following random split is only a demonstration to compare how the evaluation changes when observations from different time periods are mixed.

In [8]:
from sklearn.model_selection import train_test_split

X_random = X_train[baseline_features]
y_random = y_train

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X_random,
    y_random,
    test_size=0.3,
    random_state=42
)

random_model = LinearRegression()
random_model.fit(X_train_random, y_train_random)

random_predictions = random_model.predict(X_test_random)

random_rmse = np.sqrt(
    mean_squared_error(y_test_random, random_predictions)
)
random_mae = mean_absolute_error(y_test_random, random_predictions)
random_r2 = r2_score(y_test_random, random_predictions)

print("Random Split Demo Results")
print("-------------------------")
print("RMSE:", random_rmse)
print("MAE :", random_mae)
print("R2  :", random_r2)

Random Split Demo Results
-------------------------
RMSE: 128.22980430638523
MAE : 97.1755913113558
R2  : 0.3460524185547861


### Chronological vs Random Split

The random split gives better scores than the chronological split. The random split has an RMSE of about 128.23 compared with 191.30 for the chronological split, while its R² is about 0.346 compared with 0.242.

However, the chronological split is the main result because this dataset is time-based. A random split mixes observations from different time periods, so the model can learn from data that would not be available when making a real future prediction. Therefore, the random split can make the model performance look better than it would in a realistic forecasting situation.

In [9]:
import pandas as pd

baseline_results = pd.DataFrame({
    'dteday': test['dteday'].values,
    'hr': test['hr'].values,
    'y_true': y_test.values,
    'baseline': baseline_predictions
})

baseline_results.to_csv(
    '../results/preds_baseline.csv',
    index=False
)

display(baseline_results.head())

print("Saved results/preds_baseline.csv successfully.")

,dteday,hr,y_true,baseline
0,2012-07-01,0,149,218.190681
1,2012-07-01,1,93,219.785255
2,2012-07-01,2,90,221.379829
3,2012-07-01,3,33,229.374799
4,2012-07-01,4,4,230.969373


Saved results/preds_baseline.csv successfully.


### Limitation of the Baseline Model

The baseline Linear Regression model is simple and cannot capture all the patterns in bike demand. In particular, it treats `hr` as a normal numerical feature and assumes a simple linear relationship between hour and demand. However, bike demand can have multiple peaks during a day, such as morning and evening commuting hours. A single linear relationship with `hr` cannot represent these two peaks properly. This is one reason why the baseline model has relatively high prediction error.